In [16]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_elasticsearch import ElasticsearchStore

In [17]:
load_dotenv()

# --- Rutas ---
SEMANTIC_PATH = "/home/pablo/Documentos/ictl-rag/rag-asistent/data_procesor/semantic_chunks.json"

# --- Elasticsearch ---
ES_URL = os.environ.get("ELASTICSEARCH_URL", "http://localhost:9200")
INDEX_NAME = "rag_index"

In [18]:
# --- Cargar chunks ---
with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks cargados:", len(chunks))

Chunks cargados: 408


In [ ]:
# --- Convertir a Documentos ---

USER_ID = "default_user"

def chunks_to_documents(chunks):
    docs = []
    for chunk in chunks:
        metadata = {
            "chunk_id": chunk["chunk_id"],
            "user_id": USER_ID,
            "section_title": chunk["section_title"],
            "section_id": chunk["section_id"],
            "page": chunk["page"],
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "source": chunk["source"],
            "parent_element_id": chunk["parent_element_id"]
        }
        docs.append(Document(page_content=chunk["text"], metadata=metadata))
    return docs

documents = chunks_to_documents(chunks)
print("Documentos creados:", len(documents))

Documentos creados: 408


In [14]:
# --- Embeddings ---
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [15]:
# --- Indexar en Elasticsearch ---
vectorstore = ElasticsearchStore.from_documents(
    documents=documents,
    embedding=embeddings,
    es_url=ES_URL,
    index_name=INDEX_NAME
)

print(f"Documentos indexados en Elasticsearch: {INDEX_NAME}")

Documentos indexados en Elasticsearch: rag_index
